# Runiverse · runiverse-ai Colab 실생성 검증
- 목적: 서버 코드(app·generator·storage)를 Colab GPU에서 그대로 실행
- 런타임: 런타임 유형 변경 → GPU(T4)
- 통과 기준: `/ping` 200 → `/invocations` 200 + wav 48kHz·약 30초 → BPM 500 요청 400

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

assert Path("/content").exists(), "Google Colab에서 실행하세요."
ROOT = Path("/content/runiverse_acestep")
REPO = ROOT / "ACE-Step-1.5"
SOURCE_COMMIT = "ca1e85fe9430179831e6bc6be790c332190a3866"
REPO_URL = "https://github.com/ace-step/ACE-Step-1.5.git"
ROOT.mkdir(parents=True, exist_ok=True)
print(subprocess.run(["nvidia-smi"], check=True, text=True, capture_output=True).stdout)

In [ ]:
# ACE-Step 환경 (베이스라인 노트북 2번 셀과 동일)
def run(args, **kwargs):
    print("실행:", " ".join(map(str, args)))
    subprocess.run(list(map(str, args)), check=True, **kwargs)

run([sys.executable, "-m", "pip", "install", "--quiet", "uv>=0.8"])
UV = shutil.which("uv")
run([UV, "python", "install", "3.11"])
if not REPO.exists():
    run(["git", "clone", "--no-checkout", "--depth", "1", REPO_URL, REPO])
run(["git", "fetch", "--depth", "1", "origin", SOURCE_COMMIT], cwd=REPO)
run(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=REPO)
run([UV, "sync", "--frozen", "--no-dev", "--python", "3.11"], cwd=REPO)
PY311 = REPO / ".venv" / "bin" / "python"
print("추론용 Python:", PY311)

In [ ]:
# 모델 다운로드 + 리비전 기록 (download_models.py와 같은 위치: checkpoints/model_revisions.json)
CHECKPOINTS = REPO / "checkpoints"
CHECKPOINTS.mkdir(exist_ok=True)
download = ROOT / "download_verify.py"
download.write_text('''
import json, sys
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
target = Path(sys.argv[1])
repo_id = "ACE-Step/Ace-Step1.5"
rev_file = target / "model_revisions.json"
revisions = json.loads(rev_file.read_text()) if rev_file.exists() else {}
revisions.setdefault(repo_id, HfApi().model_info(repo_id).sha)
snapshot_download(repo_id=repo_id, revision=revisions[repo_id], local_dir=str(target))
rev_file.write_text(json.dumps(revisions, indent=2))
print("모델 준비 완료:", revisions)
''', encoding="utf-8")
run([PY311, download, CHECKPOINTS], cwd=REPO)

In [ ]:
# runiverse-ai 코드 가져오기
# A) GitHub 원격 생성 후: GITHUB_URL 지정 (비공개면 토큰 포함 URL)
# B) 그 전: Drive의 DRIVE_CODE_DIR에 저장소 폴더 업로드
GITHUB_URL = ""
DRIVE_CODE_DIR = "/content/drive/MyDrive/Runiverse/runiverse-ai"
CODE_DIR = ROOT / "runiverse-ai"
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
if GITHUB_URL:
    run(["git", "clone", "--depth", "1", GITHUB_URL, CODE_DIR])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copytree(DRIVE_CODE_DIR, CODE_DIR,
                    ignore=shutil.ignore_patterns(".venv", ".git", "__pycache__", ".model_build", ".superpowers"))
run([UV, "pip", "install", "--python", PY311, CODE_DIR])

In [ ]:
# SageMaker와 같은 /opt/ml/model 경로로 연결 후 서버 실행 (결과는 로컬 저장)
MODEL_DIR = Path("/opt/ml/model")
MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
if MODEL_DIR.is_symlink():
    MODEL_DIR.unlink()
if not MODEL_DIR.exists():
    MODEL_DIR.symlink_to(CHECKPOINTS, target_is_directory=True)

OUT_DIR = ROOT / "verify_outputs"
SERVER_LOG = ROOT / "server.log"
env = os.environ.copy()
env.update({
    "GENERATOR": "acestep", "STORAGE": "local", "LOCAL_OUTPUT_DIR": str(OUT_DIR),
    "MODEL_DIR": str(MODEL_DIR), "ACESTEP_ROOT": str(REPO), "ACESTEP_CHECKPOINTS_DIR": str(MODEL_DIR),
    "ACESTEP_COMMIT": SOURCE_COMMIT, "IMAGE_TAG": "colab", "HF_HUB_OFFLINE": "1",
    "TOKENIZERS_PARALLELISM": "false", "MPLBACKEND": "Agg", "LOG_LEVEL": "INFO",
})
server = subprocess.Popen(
    [PY311, "-m", "uvicorn", "runiverse_music.app:build_app", "--factory",
     "--host", "127.0.0.1", "--port", "8080", "--workers", "1"],
    cwd=REPO, env=env, stdout=SERVER_LOG.open("w"), stderr=subprocess.STDOUT,
)

import urllib.request
deadline = time.time() + 600
while True:
    if server.poll() is not None:
        print(SERVER_LOG.read_text()[-5000:])
        raise RuntimeError("서버가 종료되었습니다. 위 로그를 확인하세요.")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8080/ping", timeout=5) as r:
            if r.status == 200:
                break
    except OSError:
        pass
    if time.time() > deadline:
        raise TimeoutError("10분 안에 /ping 200이 오지 않았습니다.")
    time.sleep(5)
print("/ping 200 — 모델 로드 완료")

In [ ]:
# 기본 요청 → 200 + 음원 확인
import requests
from IPython.display import Audio, display

payload = {
    "request_id": "colab-verify-001",
    "caption": ("Energetic electronic running music, steady clear drum beat, "
                "uplifting synth melody, consistent rhythm, instrumental, no vocals"),
    "bpm": 170, "duration": 30.0, "seed": 42, "instrumental": True,
}
r = requests.post("http://127.0.0.1:8080/invocations", json=payload, timeout=600)
print(r.status_code)
result = r.json()
print(json.dumps(result, indent=2, ensure_ascii=False))
assert r.status_code == 200, SERVER_LOG.read_text()[-5000:]
assert result["sample_rate"] == 48000
assert abs(result["actual_duration_seconds"] - 30) <= 1
display(Audio(filename=str(OUT_DIR / result["audio_s3_key"])))

In [ ]:
# 범위 위반 → 400
bad = requests.post("http://127.0.0.1:8080/invocations", json={**payload, "bpm": 500}, timeout=60)
print(bad.status_code, bad.json())
assert bad.status_code == 400

In [ ]:
# 서버 종료 + 로그 확인 (model_loaded·generated JSON 로그)
server.terminate()
server.wait(timeout=30)
print(SERVER_LOG.read_text()[-3000:])